# Pikachu HP Prediction — GDG Recruitment Task (v2, tuned)
Predict `pikachu_hp` for each `battle_turn` in `test.csv`.

**Story so far:**
1. `trainer_focus_score` looked like a near-perfect predictor in training (corr ≈ 0.96) but turned out to be a **leaked/spurious feature** — models using it scored *worse than a flat mean baseline* on the real leaderboard (negative R²). Dropping it flipped the score to **positive (~0.645)**.
2. This version tunes the model further: feature engineering from battle mechanics, and a hyperparameter search using `HistGradientBoostingRegressor` (sklearn's fast, LightGBM-style boosting implementation).

**Covers:** data cleaning → linear/ridge/lasso → gradient descent (SGD) → polynomial regression → random forest → SVM → ensemble learning (AdaBoost, Gradient Boosting, Voting) → leak detection → feature engineering → hyperparameter tuning → final tuned model.


In [ ]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, Lasso, LogisticRegression
from sklearn.ensemble import (RandomForestRegressor, AdaBoostRegressor,
                               GradientBoostingRegressor, HistGradientBoostingRegressor,
                               VotingRegressor)
from sklearn.svm import SVR, SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

TRAIN_PATH = 'train.csv'
TEST_PATH = 'test.csv'
OUT_PATH = 'submission.csv'

TARGET = 'pikachu_hp'
ID_COL = 'battle_turn'
LEAK_COL = 'trainer_focus_score'

CAT_COLS = ['opponent_pokemon', 'opponent_type', 'move_used', 'move_type', 'move_category',
            'weather_condition', 'pikachu_status', 'terrain_type', 'held_item', 'pikachu_ability']


## 1. Load Data

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
print(f"train shape: {train.shape}   test shape: {test.shape}")

NUM_COLS = [c for c in train.columns if c not in CAT_COLS + [TARGET, ID_COL, LEAK_COL]]
train.head()


## 2. EDA — Correlation Check (this is where the leak was found)
`trainer_focus_score` has a suspiciously high correlation with the target — far higher than any genuine battle stat. That mismatch (too good on training data, useless on real battles) is the signature of a leaked feature, confirmed by leaderboard testing in v1.

In [ ]:
numeric_corr = train.select_dtypes(include=[np.number]).corr()[TARGET].sort_values(ascending=False)
print(numeric_corr)


## 3. Data Cleaning
- categorical NaN → `'None'`
- numeric NaN → median from **TRAIN only**
- **drop `trainer_focus_score`** (the leaked feature)
- one-hot encode categoricals

In [ ]:
for c in CAT_COLS:
    train[c] = train[c].fillna('None')
    test[c] = test[c].fillna('None')

for c in NUM_COLS:
    med = train[c].median()
    train[c] = train[c].fillna(med)
    test[c] = test[c].fillna(med)


## 4. Feature Engineering
Feature importances from an earlier model showed `previous_hp` and `damage_dealt` together carry ~91% of the importance — the battle mechanic is roughly `pikachu_hp ≈ previous_hp - damage_dealt + healing_applied` (clipped to `[0, max_hp]`). We add this and a couple of normalized ratios explicitly. (Note: in testing this didn't improve R² — the tree models were already combining these columns just as well — but it's kept here since it makes the model's logic more transparent and is a fair thing to try.)

In [ ]:
for df in (train, test):
    df['hp_est'] = (df['previous_hp'] - df['damage_dealt'] + df['healing_applied']).clip(0, df['max_hp'])
    df['hp_pct_prev'] = df['previous_hp'] / df['max_hp']
    df['dmg_pct'] = df['damage_dealt'] / df['max_hp']
    df['heal_pct'] = df['healing_applied'] / df['max_hp']

y = train[TARGET].copy()
test_ids = test[ID_COL]

combined = pd.concat([train.drop(columns=[TARGET, ID_COL, LEAK_COL]),
                       test.drop(columns=[ID_COL, LEAK_COL])],
                      keys=['train', 'test'])
combined = pd.get_dummies(combined, columns=CAT_COLS, drop_first=True)

X = combined.xs('train').reset_index(drop=True)
X_test = combined.xs('test').reset_index(drop=True)
X_test = X_test.reindex(columns=X.columns, fill_value=0)

print(f"features after cleaning + engineering + one-hot: {X.shape[1]}")


## 5. Train / Validation Split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_STATE
)

results = {}


def report(name, rmse, mae, r2, seconds):
    print(f"{name:32s} RMSE={rmse:7.3f}  MAE={mae:7.3f}  R2={r2:6.3f}  ({seconds:4.1f}s)")


def run_model(name, factory, scale=False, X_tr=None, y_tr=None, X_va=None, y_va=None):
    X_tr = X_train if X_tr is None else X_tr
    y_tr = y_train if y_tr is None else y_tr
    X_va = X_val if X_va is None else X_va
    y_va = y_val if y_va is None else y_va

    t0 = time.time()
    if scale:
        scaler = StandardScaler()
        X_tr_used = scaler.fit_transform(X_tr)
        X_va_used = scaler.transform(X_va)
    else:
        X_tr_used = X_tr
        X_va_used = X_va

    model = factory()
    model.fit(X_tr_used, y_tr)
    preds = model.predict(X_va_used)
    rmse = mean_squared_error(y_va, preds) ** 0.5
    mae = mean_absolute_error(y_va, preds)
    r2 = r2_score(y_va, preds)
    dt = time.time() - t0
    report(name, rmse, mae, r2, dt)

    results[name] = dict(rmse=rmse, mae=mae, r2=r2, factory=factory, scale=scale)


## 6. Simple & Multiple Linear Regression

In [ ]:
run_model("Linear Regression (OLS)", lambda: LinearRegression())


## 7. Gradient Descent (Stochastic)
`SGDRegressor` uses stochastic gradient descent instead of the closed-form normal equation.

In [ ]:
run_model("SGD Regressor (stochastic GD)",
          lambda: SGDRegressor(max_iter=2000, random_state=RANDOM_STATE),
          scale=True)


## 8. Polynomial Regression
On a handful of key numeric features to avoid a dimensionality blow-up.

In [ ]:
poly_feats = ['pikachu_level', 'opponent_level', 'damage_dealt', 'previous_hp', 'max_hp']
poly = PolynomialFeatures(degree=2, include_bias=False)
t0 = time.time()
Xtr_poly = poly.fit_transform(X_train[poly_feats])
Xva_poly = poly.transform(X_val[poly_feats])
poly_model = LinearRegression()
poly_model.fit(Xtr_poly, y_train)
preds = poly_model.predict(Xva_poly)
rmse = mean_squared_error(y_val, preds) ** 0.5
mae = mean_absolute_error(y_val, preds)
r2 = r2_score(y_val, preds)
report("Polynomial Regression (deg 2)", rmse, mae, r2, time.time() - t0)
results["Polynomial Regression (deg 2)"] = dict(rmse=rmse, mae=mae, r2=r2,
                                                 factory=None, scale=False, special='poly')


## 9. Ridge & Lasso Regression

In [ ]:
run_model("Ridge Regression", lambda: Ridge(alpha=1.0, random_state=RANDOM_STATE))
run_model("Lasso Regression", lambda: Lasso(alpha=0.1, random_state=RANDOM_STATE))


## 10. Random Forest & SVM
Trained on subsamples to keep runtime reasonable on a single CPU core.

In [ ]:
tree_sub_idx = X_train.sample(n=15000, random_state=RANDOM_STATE).index
X_train_sub = X_train.loc[tree_sub_idx]
y_train_sub = y_train.loc[tree_sub_idx]

run_model("Random Forest", lambda: RandomForestRegressor(
    n_estimators=60, max_depth=12, random_state=RANDOM_STATE, n_jobs=1),
    X_tr=X_train_sub, y_tr=y_train_sub)

sub_idx = X_train.sample(n=5000, random_state=RANDOM_STATE).index
run_model("SVR (SVM for regression)", lambda: SVR(kernel='rbf'),
          scale=True, X_tr=X_train.loc[sub_idx], y_tr=y_train.loc[sub_idx])


## 11. Ensemble Learning: AdaBoost, Gradient Boosting, Voting

In [ ]:
run_model("AdaBoost", lambda: AdaBoostRegressor(n_estimators=50, random_state=RANDOM_STATE),
          X_tr=X_train_sub, y_tr=y_train_sub)

run_model("Gradient Boosting", lambda: GradientBoostingRegressor(
    n_estimators=100, max_depth=3, random_state=RANDOM_STATE),
    X_tr=X_train_sub, y_tr=y_train_sub)

run_model("Voting Ensemble (Ridge+RF+GB)", lambda: VotingRegressor([
    ('ridge', Ridge(alpha=1.0)),
    ('rf', RandomForestRegressor(n_estimators=60, max_depth=12, random_state=RANDOM_STATE, n_jobs=1)),
    ('gb', GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=RANDOM_STATE)),
]), X_tr=X_train_sub, y_tr=y_train_sub)


## 12. Bonus: Logistic Regression, Naive Bayes & SVM Classifier
Classification algorithms, demonstrated on HP bucketed into Low/Medium/High.

In [ ]:
bins = [-1, 27, 54, 200]
labels = ['Low', 'Medium', 'High']
y_train_cls = pd.cut(y_train, bins=bins, labels=labels)
y_val_cls = pd.cut(y_val, bins=bins, labels=labels)

cls_sub_idx = X_train.sample(n=8000, random_state=RANDOM_STATE).index
scaler = StandardScaler()
Xtr_s = scaler.fit_transform(X_train.loc[cls_sub_idx])
Xva_s = scaler.transform(X_val)

for name, clf in [
    ("Logistic Regression", LogisticRegression(max_iter=1000)),
    ("Naive Bayes (Gaussian)", GaussianNB()),
    ("SVM Classifier (SVC)", SVC()),
]:
    t0 = time.time()
    clf.fit(Xtr_s, y_train_cls.loc[cls_sub_idx])
    preds = clf.predict(Xva_s)
    acc = accuracy_score(y_val_cls, preds)
    print(f"{name:32s} Accuracy={acc:.3f}  ({time.time() - t0:4.1f}s)")


## 13. Robust 5-Fold Cross-Validation (sanity check)
A single train/val split can be misleading (as the leaked feature taught us). 5-fold CV gives a stable, honest estimate before trusting any model.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for name, factory in [
    ('Ridge', lambda: Ridge(alpha=1.0)),
    ('Random Forest', lambda: RandomForestRegressor(n_estimators=60, max_depth=12, random_state=RANDOM_STATE, n_jobs=1)),
    ('Gradient Boosting', lambda: GradientBoostingRegressor(n_estimators=150, max_depth=3, random_state=RANDOM_STATE)),
]:
    scores = []
    for tr_idx, va_idx in kf.split(X):
        m = factory()
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        preds = m.predict(X.iloc[va_idx])
        scores.append(r2_score(y.iloc[va_idx], preds))
    print(f"{name:20s} CV R2 mean={np.mean(scores):.4f}  std={np.std(scores):.4f}")


## 14. Hyperparameter Tuning — HistGradientBoostingRegressor
`HistGradientBoostingRegressor` is sklearn's fast, histogram-based boosting implementation (similar idea to LightGBM/XGBoost) — much faster to train than plain `GradientBoostingRegressor`, which makes a real hyperparameter search practical. A handful of configs were tried (iterations, tree depth, leaf count, L2 regularization, loss function); the results below reproduce that search. `squared_error` loss beat `poisson` and `absolute_error`, and blending with Random Forest / Ridge did not beat the tuned model alone — this dataset appears to have a genuine noise ceiling (opponent actions aren't fully predictable from the given columns).

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
configs = [
    dict(max_iter=200, max_depth=6, learning_rate=0.1, l2_regularization=0.0),
    dict(max_iter=300, max_depth=6, learning_rate=0.05, l2_regularization=0.0),
    dict(max_iter=300, max_depth=8, learning_rate=0.05, l2_regularization=0.1),
    dict(max_iter=300, max_leaf_nodes=31, learning_rate=0.05, l2_regularization=1.0),  # winner
]
for cfg in configs:
    t0 = time.time()
    scores = []
    for tr_idx, va_idx in kf.split(X):
        m = HistGradientBoostingRegressor(random_state=RANDOM_STATE, **cfg)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        preds = m.predict(X.iloc[va_idx])
        scores.append(r2_score(y.iloc[va_idx], preds))
    print(cfg, 'CV R2 mean=%.4f std=%.4f time=%.1fs' % (np.mean(scores), np.std(scores), time.time() - t0))


## 15. Final Model — Tuned HistGradientBoostingRegressor
Best config from the search above, retrained on the full training set.

In [ ]:
final_model = HistGradientBoostingRegressor(
    max_iter=300, max_leaf_nodes=31, learning_rate=0.05,
    l2_regularization=1.0, loss='squared_error', random_state=RANDOM_STATE
)
final_model.fit(X, y)
test_preds = final_model.predict(X_test)


## 16. Build Submission File

In [ ]:
# HP can't be negative or exceed max_hp for that row
test_preds = np.clip(test_preds, 0, X_test['max_hp'].values)
test_preds = np.round(test_preds).astype(int)

submission = pd.DataFrame({ID_COL: test_ids, TARGET: test_preds})
submission.to_csv(OUT_PATH, index=False)
print(f"Saved submission -> {OUT_PATH}")
submission.head()
